In [ ]:
#This file for measures used in powerbi dashbord

Drop-off after month 1 measure

In [ ]:
import pandas as pd

txn = pd.read_csv("clean/transactions_clean.csv", parse_dates=['transaction_date'])


first_txn = txn.groupby('account_id')['transaction_date'].min().reset_index()
first_txn.columns = ['account_id', 'first_date']

txn = txn.merge(first_txn, on='account_id')
txn['days_since_first'] = (txn['transaction_date'] - txn['first_date']).dt.days


active_accounts = txn['account_id'].nunique()
month2_accounts = txn[(txn['days_since_first'] > 30) & (txn['days_since_first'] <= 60)]['account_id'].nunique()

dropoff_rate = 1 - (month2_accounts / active_accounts)
print(f"Drop-off after month 1: {dropoff_rate:.4f}")

cohort_retention Curve

In [ ]:
import pandas as pd
cohort_txn_data = pd.read_csv("clean/transactions_clean.csv", parse_dates=['transaction_date'])


cohort_first_month = cohort_txn_data.groupby('account_id')['transaction_date'].min().dt.to_period('M')
cohort_txn_period = cohort_txn_data['transaction_date'].dt.to_period('M')

cohort_txn_data['cohort_month'] = cohort_txn_data['account_id'].map(cohort_first_month)
cohort_txn_data['months_offset'] = (cohort_txn_period.astype('int64') - cohort_txn_data['cohort_month'].astype('int64'))


cohort_group_size = cohort_first_month.value_counts().sort_index()


cohort_retention_rows = []
for offset_val in range(0, 13):
    accounts_active_this_offset = cohort_txn_data[cohort_txn_data['months_offset'] == offset_val].groupby('cohort_month')['account_id'].nunique()
    retention_pct_val = (accounts_active_this_offset / cohort_group_size).mean() * 100
    cohort_retention_rows.append({'months_since_join': offset_val, 'retention_pct': round(retention_pct_val, 1)})

cohort_retention_output = pd.DataFrame(cohort_retention_rows)
print(cohort_retention_output)

cohort_retention_output.to_csv("clean/cohort_retention_curve.csv", index=False)
print("Saved!")

RFM Customer Segments

In [ ]:


rfm_txn_data = pd.read_csv("clean/transactions_clean.csv", parse_dates=['transaction_date'])
rfm_acc_data = pd.read_csv("clean/accounts_clean.csv")


rfm_txn_data = rfm_txn_data.merge(rfm_acc_data[['account_id', 'customer_id']], on='account_id')

today_ref_date = pd.Timestamp('2026-01-01')

rfm_base = rfm_txn_data.groupby('customer_id').agg(
    recency_days=('transaction_date', lambda x: (today_ref_date - x.max()).days),
    frequency=('transaction_id', 'count'),
    monetary=('amount_pkr_capped', 'sum')
).reset_index()


rfm_base['R_score'] = pd.qcut(rfm_base['recency_days'], 5, labels=[5,4,3,2,1]).astype(int)
rfm_base['F_score'] = pd.qcut(rfm_base['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm_base['M_score'] = pd.qcut(rfm_base['monetary'], 5, labels=[1,2,3,4,5]).astype(int)

def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    if r >= 4 and f >= 3:
        return 'Loyal'
    if r >= 4 and f <= 2:
        return 'New / Low Activity'
    if r == 3:
        return 'Needs Attention'
    if r <= 2 and m >= 4:
        return 'At Risk - High Value'
    return 'Hibernating'

rfm_base['segment'] = rfm_base.apply(assign_segment, axis=1)

# summary table for  Power BI 
rfm_segment_summary = rfm_base.groupby('segment').agg(
    customers=('customer_id', 'count'),
    avg_recency_days=('recency_days', 'mean'),
    total_value_pkr=('monetary', 'sum')
).round(1).reset_index()

rfm_segment_summary['pct_of_customers'] = (100 * rfm_segment_summary['customers'] / rfm_segment_summary['customers'].sum()).round(1)
rfm_segment_summary['pct_of_value'] = (100 * rfm_segment_summary['total_value_pkr'] / rfm_segment_summary['total_value_pkr'].sum()).round(1)

print(rfm_segment_summary)
rfm_segment_summary.to_csv("clean/rfm_segments.csv", index=False)
print("Saved!")